# Week 61, Day 3: contextlib

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w061d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from contextlib import contextmanager

@contextmanager
def announced(name):
    print("before", name)
    try:
        yield name.upper()
    finally:
        print("after", name)

with announced("tea") as shout:
    print("inside with", shout)

try:
    with announced("jam"):
        raise ValueError("spilt")
except ValueError as e:
    print("caught", e)

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer(results):
    start = time.perf_counter()
    try:
        yield
    finally:
        results.append(time.perf_counter() - start)

times = []
with timer(times):
    sum(range(50_000))
with timer(times):
    pass
print(len(times), all(t >= 0 for t in times))

In [ ]:
from contextlib import contextmanager

state = {"mode": "normal"}

@contextmanager
def careless():
    state["mode"] = "special"
    yield
    state["mode"] = "normal"                 # skipped if the body fails

@contextmanager
def careful():
    state["mode"] = "special"
    try:
        yield
    finally:
        state["mode"] = "normal"

for manager in (careless, careful):
    try:
        with manager():
            raise RuntimeError
    except RuntimeError:
        pass
    print(manager.__name__, state["mode"])
    state["mode"] = "normal"

In [ ]:
from contextlib import contextmanager

@contextmanager
def all_or_nothing(items):
    backup = list(items)
    try:
        yield items
    except Exception:
        items[:] = backup                  # roll back, then let the error carry on
        raise
    else:
        print("committed", items)

basket = ["tea"]
with all_or_nothing(basket) as b:
    b.append("jam")
try:
    with all_or_nothing(basket) as b:
        b.append("cake")
        raise ValueError("out of money")
except ValueError:
    pass
print(basket)

In [ ]:
from contextlib import suppress, redirect_stdout, nullcontext, closing
import io, os

with suppress(FileNotFoundError):
    os.remove("no-such-file.txt")
print("no error")

buffer = io.StringIO()
with redirect_stdout(buffer):
    print("hidden from the screen")
print("captured:", repr(buffer.getvalue()))

class Connection:
    def close(self):
        print("connection closed")

with closing(Connection()):
    pass

def first_line(source):
    manager = open(source) if isinstance(source, str) else nullcontext(source)
    with manager as f:
        return f.readline().strip()

with open("demo.txt", "w") as f:
    f.write("hello\nworld\n")
stream = io.StringIO("from a stream\n")
print(first_line("demo.txt"), "|", first_line(stream), "|", stream.closed)

In [ ]:
from contextlib import ContextDecorator

class section(ContextDecorator):
    def __init__(self, name, log):
        self.name, self.log = name, log
    def __enter__(self):
        self.log.append(f"start {self.name}")
        return self
    def __exit__(self, *exc):
        self.log.append(f"end {self.name}")
        return False

log = []

@section("report", log)
def make_report():
    log.append("writing")

make_report()
with section("manual", log):
    log.append("by hand")
print(log)

> **Key idea.** @contextmanager turns a generator into a context manager: code before the single yield is the setup, the yielded value goes to as, and code after is the cleanup. An error in the body is raised at the yield, so put the yield in try and the cleanup in finally, or the cleanup is skipped on failure. Catch around the yield to log, roll back or (rarely, specifically) suppress. Such managers are one-shot per call and must yield exactly once. contextlib also gives suppress, redirect_stdout, chdir, closing and nullcontext, and ContextDecorator (like @contextmanager managers) lets a manager decorate whole functions.

### Exercise: Generator managers

Write with `@contextmanager`: `tag(log, name)` appends `"<name>"` to the list on entry and `"</name>"` on exit (even on error), yielding nothing; and `counting(counter, key)`: adds 1 to `counter[key]` (a dict, starting from 0 if missing) on entry, yields the new count, and subtracts 1 on exit, whatever happened.

In [ ]:
from contextlib import contextmanager

@contextmanager
def tag(log, name):
    ___

@contextmanager
def counting(counter, key):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gen():\n    \"contextmanager\"\n    log = []\n    with tag(log, \"html\"):\n        with tag(log, \"body\"):\n            log.append(\"text\")\n    assert log == [\"<html>\", \"<body>\", \"text\", \"</body>\", \"</html>\"]\n    log = []\n    try:\n        with tag(log, \"p\"):\n            raise ValueError\n    except ValueError:\n        pass\n    assert log == [\"<p>\", \"</p>\"]\n    c = {}\n    with counting(c, \"users\") as n:\n        assert n == 1\n        with counting(c, \"users\") as m:\n            assert m == 2\n    assert c == {\"users\": 0}\n    try:\n        with counting(c, \"x\"):\n            raise KeyError\n    except KeyError:\n        pass\n    assert c[\"x\"] == 0")

### Exercise: Working in another folder

Write `inside(path)` with `@contextmanager`: create the folder (and any missing parents) if needed, change into it, yield the absolute path of that folder, and change back to the previous folder afterwards, even on an error. Use `os.makedirs` with `exist_ok=True`, `os.getcwd` and `os.chdir`.

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def inside(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\ndef test_folder():\n    \"changing folder\"\n    start = os.getcwd()\n    with inside(\"work/reports\") as where:\n        assert os.getcwd() == where and where.endswith(os.path.join(\"work\", \"reports\"))\n        with open(\"r.txt\", \"w\") as f:\n            f.write(\"x\")\n    assert os.getcwd() == start and os.path.exists(os.path.join(\"work\", \"reports\", \"r.txt\"))\n    try:\n        with inside(\"work\"):\n            raise RuntimeError\n    except RuntimeError:\n        pass\n    assert os.getcwd() == start")

### Exercise: Capturing printed output

Write `printed(func, *args)`: call the function with `redirect_stdout` into an `io.StringIO` and return a pair `(result, text_printed)`. Then `quiet(func, *args)`: call it with its printed output thrown away (redirect to a fresh `StringIO`) and return only the result.

In [ ]:
import io
from contextlib import redirect_stdout

def printed(func, *args):
    ___

def quiet(func, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_capture():\n    \"redirect_stdout\"\n    def noisy(a, b):\n        print(\"adding\", a, b)\n        return a + b\n    assert printed(noisy, 2, 3) == (5, \"adding 2 3\\n\") and quiet(noisy, 1, 1) == 2\n    assert printed(lambda: None) == (None, \"\")\n    import contextlib\n    assert \"redirect_stdout(\" in source")

### Exercise: A path or an open file

Write `count_lines(source)`: the number of lines in `source`, which is either a path (open it, and close it afterwards) or an already open file-like object (use it, and leave it **open**: it belongs to the caller). Use `nullcontext` so there's only one `with` statement. Then `remove_quietly(path)`: delete a file if it exists, with `contextlib.suppress`, returning True if a file was removed.

In [ ]:
import os
from contextlib import nullcontext, suppress

def count_lines(source):
    ___

def remove_quietly(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io, os\ndef test_either():\n    \"nullcontext and suppress\"\n    with open(\"three.txt\", \"w\") as f:\n        f.write(\"a\\nb\\nc\\n\")\n    s = io.StringIO(\"x\\ny\\n\")\n    assert count_lines(\"three.txt\") == 3 and count_lines(s) == 2 and not s.closed\n    assert remove_quietly(\"three.txt\") is True and not os.path.exists(\"three.txt\") and remove_quietly(\"three.txt\") is False\n    assert source.count(\"with \") == 2 and \"nullcontext(\" in source and \"suppress(\" in source")

### Exercise: With or as a decorator

Write a class `recorded(ContextDecorator)` taking a name and a list: entering appends `("start", name)`, exiting appends `("ok", name)` or `("failed", name, error_type_name)` and lets errors carry on. It must work as a `with` and as a decorator (on any function). Then a `@contextmanager` function `limited(seconds, clock)` that, on exit, raises `TimeoutError(f"took {elapsed:.1f}s")` if the block took longer than `seconds` by the given clock (it must work as a `with` and, being made by `@contextmanager`, as a decorator).

In [ ]:
from contextlib import ContextDecorator, contextmanager

class recorded(ContextDecorator):
    def __init__(self, name, log):
        ___

@contextmanager
def limited(seconds, clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deco():\n    \"ContextDecorator\"\n    log = []\n    @recorded(\"job\", log)\n    def job(x):\n        return x * 2\n    assert job(4) == 8 and job.__name__ == \"job\"\n    try:\n        with recorded(\"manual\", log):\n            raise KeyError\n    except KeyError:\n        pass\n    assert log == [(\"start\", \"job\"), (\"ok\", \"job\"), (\"start\", \"manual\"), (\"failed\", \"manual\", \"KeyError\")]\n    ticks = iter([0.0, 0.5, 10.0, 13.0])\n    clock = lambda: next(ticks)\n    with limited(1, clock):\n        pass\n    @limited(2, clock)\n    def slow():\n        return \"done\"\n    try:\n        slow()\n        assert False\n    except TimeoutError as e:\n        assert str(e) == \"took 3.0s\"")

### Bug hunt: Cleanup skipped on error

The cleanup comes after a plain `yield`, so when the body raises, the exception arrives **at** the yield and the cleanup never runs: the lock stays held and this prints `True`. Wrap the yield in `try` and `finally` so it prints `False`.

In [ ]:
from contextlib import contextmanager

state = {"locked": False}

@contextmanager
def locked():
    state["locked"] = True
    yield
    state["locked"] = False

try:
    with locked():
        raise RuntimeError
except RuntimeError:
    pass
print(state["locked"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"try and finally\"\n    expect_output(\"False\")")

### Bug hunt: Yielding twice

A generator-based manager must yield **exactly once**. This one yields in a loop, so leaving the block raises `RuntimeError: generator didn't stop`, and it prints `using 1` and then `RuntimeError`. Yield once so it prints only `using 1`.

In [ ]:
from contextlib import contextmanager

@contextmanager
def numbered():
    for n in (1, 2):
        yield n

try:
    with numbered() as n:
        print("using", n)
except RuntimeError:
    print("RuntimeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one yield\"\n    expect_output(\"using 1\")")

### Bug hunt: Reusing a one-shot manager

A `@contextmanager` object can be used only once: its generator is finished after the first `with`, so the second `with` fails (the exact exception differs between Python versions) and this prints `ok` then `failed`. Call the function again for each `with` so it prints `ok` twice.

In [ ]:
from contextlib import contextmanager

@contextmanager
def step():
    yield "ok"

manager = step()
try:
    for _ in range(2):
        with manager as m:
            print(m)
except Exception:
    print("failed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a fresh manager each time\"\n    expect_output(\"ok\\nok\")")

### Puzzle: A generator-based manager

Arrange the lines of a `@contextmanager` function that sets a flag, yields, and always clears it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # where an exception from the with body appears inside a @contextmanager generator: "at the yield" or "at the start"
q = ___   # the error from a @contextmanager generator that yields twice, as a string
r = ___   # True or False: nullcontext(5).__enter__() returns 5
s = ___   # True or False: contextlib.suppress(KeyError) also hides a ValueError
t = ___   # the Python version that added contextlib.chdir, as "3.11", "3.9" or "3.6"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    from contextlib import nullcontext\n    assert p == \"at the yield\" and q == \"RuntimeError\" and r is True, \"p, q and r\"\n    assert s is False and t == \"3.11\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: writing files atomically

A program that crashes half-way through writing a file can leave it corrupted, with neither the old contents nor the new. The safe pattern: write to a **temporary** file in the same folder, and only when everything has been written, **replace** the real file with it in one step (`os.replace` does this atomically on the same disk). If anything fails, delete the temporary file and leave the original untouched.

- `atomic_write(path)`: a `@contextmanager` that yields an open file to write to (the temporary file, named `path + ".tmp"`), and on success closes it and replaces `path` with it; on an error closes and removes the temporary file and lets the error carry on.
- `update_lines(path, func)`: read the file's lines (if it exists; otherwise start with none), write `func(lines)` back atomically, one line each, and return the number of lines written.
- `backup_then_write(path, text)`: if the file exists, keep a copy as `path + ".bak"` (atomically too), then write the text atomically.

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def atomic_write(path):
    ___

def update_lines(path, func):
    ___

def backup_then_write(path, text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\ndef test_atomic():\n    \"temp file and replace\"\n    with atomic_write(\"data.txt\") as f:\n        f.write(\"version 1\\n\")\n    with open(\"data.txt\") as f:\n        assert f.read() == \"version 1\\n\"\n    try:\n        with atomic_write(\"data.txt\") as f:\n            f.write(\"half of version 2\")\n            raise RuntimeError(\"crash\")\n    except RuntimeError:\n        pass\n    with open(\"data.txt\") as f:\n        assert f.read() == \"version 1\\n\"\n    assert not os.path.exists(\"data.txt.tmp\")\n    assert update_lines(\"list.txt\", lambda lines: lines + [\"first\"]) == 1\n    assert update_lines(\"list.txt\", lambda lines: [l.upper() for l in lines] + [\"second\"]) == 2\n    with open(\"list.txt\") as f:\n        assert f.read() == \"FIRST\\nsecond\\n\"\n    try:\n        update_lines(\"list.txt\", lambda lines: 1 / 0)\n    except ZeroDivisionError:\n        pass\n    with open(\"list.txt\") as f:\n        assert f.read() == \"FIRST\\nsecond\\n\"\n    backup_then_write(\"list.txt\", \"new\\n\")\n    with open(\"list.txt\") as f, open(\"list.txt.bak\") as g:\n        assert f.read() == \"new\\n\" and g.read() == \"FIRST\\nsecond\\n\"")

## Warm-up

## A generator as a manager

`@contextmanager` (from `contextlib`, by Nick Coghlan and others, with the `with` statement in Python 2.5) wraps a generator function so that calling it gives a context manager:

- `__enter__` runs the generator up to its **single** `yield`, and returns the yielded value (bound by `as`).
- `__exit__` resumes the generator after the `yield`. On success it's resumed normally; on an error, the exception is **raised at the `yield`** inside the generator, as if the `yield` line itself had failed.

So the setup, the value and the cleanup sit together in one short function, in the order they happen: it's yesterday's classes with the boilerplate removed. Under the hood it uses Week 59's generator machinery (`next` to start, `throw` to deliver an exception) and Week 60's decorators.

## The finally rule

Because an error in the body is raised **at the `yield`**, any cleanup written after the `yield` without protection is **skipped** when the body fails, the very case context managers exist for. Always put the `yield` inside `try` and the cleanup in `finally` (or use `except`/`else` when cleanup should differ between success and failure).

Two other rules: the generator must yield **exactly once** (yielding nothing gives `RuntimeError: generator didn't yield`; yielding twice gives `RuntimeError: generator didn't stop`), and a manager made this way is **one-shot** per call: call the function again for each `with`.

## Errors inside

To react to an error, catch it around the `yield`:

- **Log and re-raise**: `except Exception: log(...); raise`.
- **Roll back**: undo the changes in `except`, then `raise`.
- **Suppress**: catch it and **don't** re-raise; the `with` block then counts as having finished normally. As before, suppress only specific exception types.

`try`/`except`/`else`/`finally` all work as usual: `else` for "the body succeeded", `finally` for "always".

## Ready-made helpers

`contextlib` already has the common ones:

| Helper | Does |
|---|---|
| `suppress(*types)` | ignores those exception types inside the block (Tuesday's `Ignoring`) |
| `redirect_stdout(stream)` / `redirect_stderr(stream)` | sends `print` output to another stream, such as an `io.StringIO`, for the block |
| `chdir(path)` | changes the current folder for the block (Python 3.11) |
| `closing(thing)` | calls `thing.close()` at the end, for objects with `close()` but no `with` support |
| `nullcontext(value)` | does nothing, giving `value` to `as`: for "use a manager only sometimes" |

`nullcontext` is handy when code takes **either** a path (which it must open and close) or an open file (which belongs to the caller and must be left open): `manager = open(path) if isinstance(x, str) else nullcontext(x)`, then one `with` either way.

## Managers as decorators

Often you want "run this whole function inside the manager". `contextlib.ContextDecorator` is a base class: a manager class that inherits from it can be used as a decorator too, wrapping each call in `with self:`. Managers made with `@contextmanager` already support this. It suits managers that don't need to hand anything to the body (timers, logging sections, temporary settings), because a decorated function can't receive the `as` value.

## Practice

## Quiz

## Daily challenge

## Before you go

Rewrite yesterday's `Patched` or `Transaction` class as a `@contextmanager` function, and compare the two for length and clarity. Tomorrow: `ExitStack`, for when you don't know in advance how many managers you'll need.